# Exploratory Data Analysis

Univariate and bivariate exploration of the KPIs built in the previous notebooks, including the funnel-level KPIs (step conversion rate, time per step). Sets up intuition for the formal hypothesis tests in `hypothesis_testing.ipynb`.

**Sections**
1. Setup
2. Univariate — client-level KPIs
3. Univariate — visit-level KPIs
4. Univariate — demographics & experiment assignment
5. Bivariate — KPIs vs. Variation (A/B preview)
6. Bivariate — demographics vs. Variation (balance check)
7. Bivariate — KPIs vs. demographics
8. Funnel KPIs — overall and by Variation
9. Time per step — overall and by Variation
10. Summary of findings

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_events = pd.read_csv("events_kpi_table.csv", parse_dates=["date_time"])
df_visits = pd.read_csv("visits_kpi_table.csv", parse_dates=["visit_start", "visit_end"])
df_clients = pd.read_csv("client_kpi_table.csv")

## Univariate — client-level KPIs

Look at the distribution, central tendency, and missingness of each client-level KPI before doing any group comparison. This step also informs which test to use later (normal-ish → t-test, skewed → non-parametric).

## Univariate — visit-level KPIs

The visit-level distributions are informative because they're the raw material that gets averaged into the client-level KPIs.

## Univariate — demographics & experiment assignment

Quick view of the population we're analyzing, plus the Test/Control split.

## Bivariate — KPIs vs. Variation (A/B preview)

This is the core comparison. Each KPI is compared between Test and Control groups. Visual differences here preview what the hypothesis tests in the next notebook will formalize.

## Bivariate — demographics vs. Variation (balance check)

If Test and Control differ systematically on demographics (age, balance, tenure, etc.), any KPI difference might be confounded by that imbalance rather than reflecting the variation itself. This is a critical check before trusting A/B test results.

## Bivariate — KPIs vs. demographics

Secondary insights: do completion rate and other KPIs vary by age, gender, or balance? Useful for context, even if not the primary research question.

## Funnel KPIs — overall and by Variation

Step conversion rate: of the visits that reached step N, what fraction moved on to step N+1? This shows where users drop off most in the funnel.

In [3]:
step_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

funnel = df_events.groupby('process_step')['visit_id'].nunique().reindex(step_order).reset_index(name='n_visits')

funnel['conversion_rate'] = funnel['n_visits'] / funnel['n_visits'].shift(1)

funnel

,process_step,n_visits,conversion_rate
0,start,144902,NaN
1,step_1,119255,0.823005
2,step_2,104341,0.874940
3,step_3,95093,0.911368
4,confirm,89826,0.944612


## Time per step — overall and by Variation

Average time users spend on each step. `confirm` is excluded because there's no event after it (its duration would be unreliable).

In [6]:
time_per_step = df_events[df_events["process_step"] != "confirm"].groupby('process_step')['time_on_each_step'].mean().reset_index()

time_per_step

,process_step,time_on_each_step
0,start,139.024150
1,step_1,39.439692
2,step_2,45.788994
3,step_3,99.382272


## Summary of findings